# Object Indexing (CCL) and Geometric Feature Extraction

This project documents the transition from raw binary image segmentation to object-level analysis of localized structures. A core challenge in vision systems is assigning unique identifiers to pixel groupings (Connected Component Labeling — CCL), enabling their isolation and parameterization.

The implementation covers:
- **Two-pass CCL algorithm:** Custom low-level labeling with a **Union-Find** disjoint-set structure, resolving merge conflicts in complex topologies (e.g., U- or W-shaped regions).
- **Benchmarking:** Performance comparison of the custom solution against the optimized `cv2.connectedComponents` implementation.
- **Shape-descriptor classification:** Extraction of translation-, scale-, and rotation-invariant geometric moments (Hu moments).
- **Real-world data analysis:** A pipeline combining thresholding, morphological opening, and final classification on noisy physical-scene images.

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
import glob

# Download test environment
files = ['ccl1.png', 'shapes.png', 'shapesReal.png']
for f in files:
    if not os.path.exists(f):
        !wget -q https://raw.githubusercontent.com/vision-agh/poc_sw/master/13_CCL/{f} --no-check-certificate

## 1. Low-Level CCL Indexing (Two-Pass Algorithm with Union-Find)

The solution iterates over the image twice. In the first pass, it scans the local neighborhood (left/up) and assigns provisional labels. When two different labels belong to the same physical shape, the conflict is recorded in a **Union-Find** graph. The second pass performs global reindexing based on computed roots from the graph.

In [ ]:
c1 = cv2.imread('ccl1.png', cv2.IMREAD_GRAYSCALE)
c1 = (c1 > 0).astype(int)
h, w = c1.shape

# Initialize Union-Find structure
L = 1
id_array = np.arange(100)

def root(i, id_arr):
    """Recursive/iterative lookup of a label root."""
    while i != id_arr[i]: 
        i = id_arr[i]
    return i

def union(p, q, id_arr):
    """Record a conflict (merge two subgraphs into one object)."""
    id_arr[root(p, id_arr)] = root(q, id_arr)

out = np.zeros((h, w), int)

# Pass 1: Assign provisional labels and build the equivalence graph
for i in range(1, h-1):
    for j in range(1, w-1):
        if c1[i, j] != 0:
            nbs = [out[i-1, j-1], out[i-1, j], out[i-1, j+1], out[i, j-1]]
            nz = [n for n in nbs if n > 0]
            
            if not nz:
                out[i, j] = L
                L += 1
            else:
                mn = min(nz)
                out[i, j] = mn
                for v in nz:
                    if v != mn:
                        union(mn, v, id_array)

# Build LUT mapping labels to flattened roots
lut = np.zeros(100, int)
for i in range(1, 100):
    lut[i] = root(i, id_array)

# Pass 2: Reindex and unify objects
res = np.zeros((h, w), int)
for i in range(h):
    for j in range(w):
        if out[i, j] > 0:
            res[i, j] = lut[out[i, j]]

plt.figure(figsize=(8,6))
plt.imshow(res, cmap='nipy_spectral')
plt.title("Custom Two-Pass CCL with Union-Find")
plt.axis('off')
plt.show()

## 2. Optimized Indexing (OpenCV API)
Validation of the custom algorithm against the highly optimized (C++ compiled) OpenCV library.

In [ ]:
_, ccl_cv2 = cv2.connectedComponents(cv2.imread('ccl1.png', cv2.IMREAD_GRAYSCALE))

plt.figure(figsize=(8,6))
plt.imshow(ccl_cv2, cmap='nipy_spectral')
plt.title("Production Implementation (cv2.connectedComponents)")
plt.axis('off')
plt.show()

## 3. Feature Extraction and Classification (Geometric Moments)

Shape discrimination is based on Hu invariants — statistical parameters computed from object pixels that are invariant to:
- Translation
- Scale
- In-plane 2D rotation

The experiment below uses the first Hu moment to select a specific geometric form (cross) from a synthetically generated dataset.

In [ ]:
sh = cv2.imread('shapes.png', cv2.IMREAD_GRAYSCALE)
_, ccl = cv2.connectedComponents(sh)
res_sh = np.zeros_like(sh)

# Feature-space analysis for each isolated object
for s in range(1, ccl.max() + 1):
    I = (ccl == s).astype(np.uint8)
    mom = cv2.moments(I)
    hu = cv2.HuMoments(mom).flatten()
    
    # Simple spatial classifier (first Hu invariant > 0.18 isolates crosses)
    if hu[0] > 0.18:
        res_sh[I == 1] = 255

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(sh, cmap='gray')
ax[0].set_title("Input Image (Test Set)")
ax[0].axis('off')
ax[1].imshow(res_sh, cmap='gray')
ax[1].set_title("Logical Classification (Isolated Objects)")
ax[1].axis('off')
plt.show()

## 4. Real-World Data Analysis (Noisy Scenes)

Applying the methodology to a physical scene requires prior preprocessing:
1. **Binary thresholding** extracts dark objects from a bright background.
2. **Morphological opening** removes dust specks and light reflections (false positives).
3. **Connected Components with Stats** provides metadata used to reject structures with area below a critical threshold (Area < 100px).

In [ ]:
shr = cv2.imread('shapesReal.png', cv2.IMREAD_GRAYSCALE)
_, thr = cv2.threshold(shr, 60, 255, cv2.THRESH_BINARY_INV)
# Filter structural noise from the background
thr = cv2.morphologyEx(thr, cv2.MORPH_OPEN, np.ones((5,5), np.uint8))

n, ccl, stats, cents = cv2.connectedComponentsWithStats(thr)

ccl_vis = ccl.copy()
res_real = np.zeros_like(shr)

for s in range(1, n):
    # Ignore speckle noise on the matrix
    if stats[s, cv2.CC_STAT_AREA] < 100: 
        continue
    
    cv2.putText(ccl_vis, str(s), (int(cents[s,0]), int(cents[s,1])), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, 255, 1)
    
    I = (ccl == s).astype(np.uint8)
    mom = cv2.moments(I)
    hu = cv2.HuMoments(mom).flatten()
    
    # Shape-geometry classifier
    if hu[0] > 0.18:
        res_real[I == 1] = 255

f, ax = plt.subplots(1, 2, figsize=(14, 7))
ax[0].imshow(ccl_vis, cmap='nipy_spectral')
ax[0].set_title("Labeling with Centroid Annotation")
ax[0].axis('off')
ax[1].imshow(res_real, cmap='gray')
ax[1].set_title("Pattern Recognition on Real-World Data")
ax[1].axis('off')
plt.show()

### Workspace Cleanup

In [ ]:
for f in files:
    try:
        os.remove(f)
    except OSError:
        pass
print("Temporary images removed from the local filesystem.")